In [1]:
4096/8

512.0

In [2]:
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-1.5B-Instruct"
max_seq_length = (
    1024  # Can increase for longer outputs, or decrease if running into OOM
)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,  # set to True for low precision training to save VRAM
    full_finetuning = False,  # set to False for LoRA training
    offload_embedding = True,  # Reduces VRAM a little
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/hero/ML_journey/random_code/unsloth/nemo_sudoku/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not patch trl.trainer.grpo_trainer: Direct module loading failed for UnslothGRPOTrainer: Unexpected optimization option triton.enable_persistent_tma_matmul, known options are ['TYPE_CHECKING', 'enable_auto_functionalized_v2', 'debug', 'disable_progress', 'verbose_progress', 'fx_graph_cache', 'fx_graph_remote_cache', 'autotune_local_cache', 'autotune_remote_cache', 'force_disable_caches', 'sleep_sec_TESTING_ONLY', 'custom_op_default_layout_constraint', 'cpp_wrapper', 'abi_compatible', 'c_shim_version', 'dce', 'static_weight_shapes', 'size_asserts', 'nan_asserts', 'pick_loop_orders', 'inplace_buffers', 'allow_buffer_reuse', 'memory_planning', 'memory_pool', 'benchmark_harness', 'epilogue_fusion', 'epilogue_fusion_first', 'pattern_matcher', 'b2b_gemm_pass', 'post_grad_custom_pre_pass', 'post_grad_custom_post_pass', 'joint_custom_pre_pass', 'joint_custom_post_pass', 'pre_grad_custom_pass', '_pre_fusion_custom_pass', 'split_cat_fx_passes', 'efficient_

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4090 Laptop GPU. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.9. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
lora_rank = 4 # Larger rank = smarter, but slower
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 42,
)

Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
import subprocess
import os
import time
import atexit
import requests

GYM_DIR = os.path.expanduser("~/Gym")

# Detect Google Colab
try:
    import google.colab
    _on_colab = True
except ImportError:
    _on_colab = False

# Step 1: Clone NeMo Gym
if not os.path.exists(GYM_DIR):
    print("Cloning NeMo Gym...")
    subprocess.run(
        ["git", "clone", "https://github.com/NVIDIA-NeMo/Gym.git", GYM_DIR],
        check=True,
    )

# Step 2: Create venv and install dependencies
if not os.path.exists(os.path.join(GYM_DIR, ".venv", "bin", "python")):
    print("Setting up NeMo Gym environment (this may take a few minutes)...")
    subprocess.run(["uv", "self", "update"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.12"], cwd=GYM_DIR, check=True)
    subprocess.run(
        [
            "bash", "-c",
            "source .venv/bin/activate && "
            "uv sync --extra dev --group docs && "
            "uv pip install reasoning-gym matplotlib pillow"
        ],
        cwd=GYM_DIR,
        check=True,
    )

# Step 3: Create dataset
_sudoku_ds = os.path.join(
    GYM_DIR, "resources_servers/reasoning_gym/data/train_mini_sudoku.jsonl"
)
if not os.path.exists(_sudoku_ds):
    print("Creating mini_sudoku dataset (2000 examples)...")
    subprocess.run(
        [
            "bash", "-c",
            "source .venv/bin/activate && python "
            "resources_servers/reasoning_gym/scripts/create_dataset.py "
            "--task mini_sudoku --size 2000 --seed 42 "
            f"--output {_sudoku_ds}",
        ],
        cwd=GYM_DIR,
        check=True,
    )

# Start NeMo Gym server if not already running
try:
    requests.get("http://127.0.0.1:11000/global_config_dict_yaml", timeout=2)
    print("NeMo Gym server already running on port 11000.")
except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
    _colab_flag = " +uv_pip_set_python=true" if _on_colab else ""
    print("Starting NeMo Gym server...")
    _ng_log = open(os.path.join(GYM_DIR, "ng_run.log"), "w")
    ng_process = subprocess.Popen(
        [
            "bash", "-c",
            "source .venv/bin/activate && ng_run "
            '"+config_paths=[resources_servers/reasoning_gym/configs/resources_only.yaml]"'
            + _colab_flag,
        ],
        cwd=GYM_DIR,
        stdout=_ng_log,
        stderr=subprocess.STDOUT,
    )

    def _cleanup_ng():
        if ng_process.poll() is None:
            ng_process.terminate()
            try:
                ng_process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                ng_process.kill()
        _ng_log.close()

    atexit.register(_cleanup_ng)

    print("Waiting for server", end="", flush=True)
    for _ in range(120):
        try:
            requests.get("http://127.0.0.1:11000/global_config_dict_yaml", timeout=2)
            break
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
            if ng_process.poll() is not None:
                raise RuntimeError(
                    "Server process exited unexpectedly. "
                    f"Check {GYM_DIR}/ng_run.log for details."
                )
            print(".", end="", flush=True)
            time.sleep(3)
    else:
        raise RuntimeError("NeMo Gym server did not start within 6 minutes.")

    print("\nHead server ready!")

Starting NeMo Gym server...
Waiting for server...
Head server ready!


In [5]:
import os
import json
from datasets import Dataset

dataset_path = "~/Gym/resources_servers/reasoning_gym/data/train_mini_sudoku.jsonl"
dataset_path = os.path.expanduser(dataset_path)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}. "
        "Run the setup cell above first."
    )

train_data = []
max_length_seen = 0
with open(dataset_path, "r") as f:
    for line in f:
        data = json.loads(line)

        # extract prompt from nemo gym format
        task_prompt = data["responses_create_params"]["input"][0]["content"]

        train_data.append(
            {
                "prompt": [{"role": "user", "content": task_prompt}],
                "answer": data["answer"],
                "metadata": data["metadata"],
            }
        )

        prompt_length = len(
            tokenizer.apply_chat_template(
                [{"role": "user", "content": task_prompt}], add_generation_prompt = True
            )
        )
        max_length_seen = max(max_length_seen, prompt_length)

print(f"Loaded {len(train_data)} examples!\n\n")
print(f"Example prompt:\n\n{train_data[0]['prompt'][0]['content']}")
train_dataset = Dataset.from_list(train_data)

Loaded 2000 examples!


Example prompt:

In 4x4 Mini Sudoku:
- Each row must contain each number from 1-4 exactly once
- Each column must contain each number 1-4 exactly once
- Each 2x2 subgrid must contain each number 1-4 exactly once
Solve this 4x4 Mini Sudoku puzzle:
4 _ _ _
_ 3 _ _
_ 1 3 _
_ _ _ _
Format your response as the puzzle above, with spaces separating each number within a row, and newlines separating rows.



In [6]:
import numpy as np


def reward_fn(completions, prompts = None, **kwargs):
    answers = kwargs["answer"]
    metadatas = kwargs["metadata"]
    scores = []
    for i, completion in enumerate(completions):
        completion_text = completion[0]["content"]
        task_prompt = prompts[i][0]["content"]

        # prepare data in NeMo Gym verifier request format
        verify_request = {
            "responses_create_params": {
                "input": [{"role": "user", "content": task_prompt, "type": "message"}]
            },
            "response": {
                "id": "resp",
                "created_at": 0,
                "model": model_name,
                "object": "response",
                "output": [
                    {
                        "id": "msg",
                        "role": "assistant",
                        "type": "message",
                        "status": "completed",
                        "content": [
                            {
                                "type": "output_text",
                                "text": completion_text,
                                "annotations": [],
                            }
                        ],
                    }
                ],
                "parallel_tool_calls": True,
                "tool_choice": "auto",
                "tools": [],
            },
            "question": task_prompt,
            "answer": answers[i],
            "metadata": metadatas[i],
        }
        try:
            # send verify request to NeMo Gym resources server
            resp = requests.post(verify_endpoint, json = verify_request, timeout = 30)
            reward = resp.json().get("reward", 0.0) if resp.status_code == 200 else 0.0
        except requests.exceptions.RequestException as e:
            print(f"Warning: verify request failed: {e}")
            reward = 0.0
        scores.append(reward)
    return np.array(scores)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

small_train_dataset = train_dataset.select(range(64))

training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=1e-5,
    weight_decay=0.001,
    warmup_ratio=0.0,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=min(max_prompt_length, 256),
    max_completion_length=128,
    num_train_epochs=1,
    max_steps=5,
    save_steps=5,
    report_to="none",
    output_dir="outputs_debug",
    epsilon_high=0.28,
    mask_truncated_completions=True,
    log_completions=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=small_train_dataset,
)

In [9]:
from transformers import TextStreamer

prompt = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": """In 4x4 Mini Sudoku:
- Each row must contain each number from 1-4 exactly once
- Each column must contain each number 1-4 exactly once
- Each 2x2 subgrid must contain each number 1-4 exactly once
Solve this 4x4 Mini Sudoku puzzle:
4 _ _ _
_ 3 _ _
_ 1 3 _
_ _ _ _
Format your response as the puzzle above, with spaces separating each number within a row, and newlines separating rows.
""",
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

def run_test(title):
    print(f"\n===== {title} =====\n")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    _ = model.generate(
        **inputs,
        do_sample=True,
        temperature=1.0,
        max_new_tokens=256,
        streamer=TextStreamer(tokenizer, skip_prompt=False),
    )
    print("\n")

run_test("BEFORE TRAINING")




===== BEFORE TRAINING =====

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
In 4x4 Mini Sudoku:
- Each row must contain each number from 1-4 exactly once
- Each column must contain each number 1-4 exactly once
- Each 2x2 subgrid must contain each number 1-4 exactly once
Solve this 4x4 Mini Sudoku puzzle:
4 _ _ _
_ 3 _ _
_ 1 3 _
_ _ _ _
Format your response as the puzzle above, with spaces separating each number within a row, and newlines separating rows.
<|im_end|>
<|im_start|>assistant
4 _ _ _
_ 3 _ _
_ 1 3 _
_ _ _ _
Here is the solved puzzle:

4 2 5 1
3 6 7 2
8 9 4 5
7 1 3 6<|im_end|>




In [10]:
trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 64
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 64 x 1) = 64
 "-____-"     Trainable parameters = 4,616,192 of 1,548,330,496 (0.30% trained)


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
run_test("AFTER TRAINING")